# Transformer-Based rPPG — Improved Pipeline

Remote Photoplethysmography (rPPG) using a CNN spatial encoder + Transformer temporal model  
with multi-task learning (waveform + BPM).

**Improvements over baseline:**
- Fixed FFT frequency indexing bug in `compute_bpm`
- Added PositionalEncoding to Transformer
- Added temporal convolution before Transformer
- Added BatchNorm + Dropout to CNN backbone
- Added BPM head back into the model (multi-task loss)
- Fixed GradScaler API for newer PyTorch
- Fixed DataLoader `num_workers=0` for Windows/macOS compatibility
- Added gradient clipping
- Added full validation loop with metrics
- Added ReduceLROnPlateau scheduler step

## 1. Installation

In [1]:
# Uncomment and run once to install dependencies
# !pip install torch torchvision torchaudio
# !pip install opencv-python mediapipe scipy scikit-learn matplotlib tqdm

## 2. Imports

In [2]:
import os
import cv2
import math
import torch
import random
import numpy as np
import scipy.signal as sp_signal
import matplotlib.pyplot as plt

from tqdm import tqdm
from scipy.fft import rfft, rfftfreq
from sklearn.model_selection import train_test_split

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import mediapipe as mp

2026-05-07 14:01:09.093113: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-07 14:01:09.095206: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-07 14:01:09.140847: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-07 14:01:09.813792: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


## 3. Configuration

In [3]:
# ================================
# IMPROVED CONFIG
# ================================

class ImprovedConfig:

    DATASET_PATH   = "/home/dhruv/Documents/rppg project/UBFC/dataset 2"
    PREPROCESS_DIR = "./preprocessed"

    FPS      = 30
    IMG_SIZE = 72
    SEQ_LEN  = 160

    BATCH_SIZE = 4
    EPOCHS     = 30
    LR         = 1e-4

    D_MODEL    = 64
    NHEAD      = 4
    NUM_LAYERS = 2

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


cfg = ImprovedConfig()
os.makedirs(cfg.PREPROCESS_DIR, exist_ok=True)

print("Using Device:", cfg.DEVICE)

Using Device: cpu


## 4. Face ROI Extraction & Signal Utilities

In [4]:
# ================================
# MEDIAPIPE FACE DETECTOR
# ================================

mp_face = mp.solutions.face_detection

face_detector = mp_face.FaceDetection(
    model_selection=0,
    min_detection_confidence=0.6
)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [5]:
def extract_face_roi(frame):
    """Detect face and return resized ROI, or None if no face found."""

    h, w, _ = frame.shape

    rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = face_detector.process(rgb)

    if not result.detections:
        return None

    det  = result.detections[0]
    bbox = det.location_data.relative_bounding_box

    x  = max(0, int(bbox.xmin  * w))
    y  = max(0, int(bbox.ymin  * h))
    bw = min(int(bbox.width  * w), w - x)
    bh = min(int(bbox.height * h), h - y)

    roi = frame[y:y + bh, x:x + bw]

    if roi.size == 0:
        return None

    roi = cv2.resize(roi, (cfg.IMG_SIZE, cfg.IMG_SIZE))

    return roi

In [6]:
def bandpass_filter(signal_data, low=0.7, high=4.0, fs=30):
    """Butterworth bandpass filter for the physiological heart-rate band (0.7–4.0 Hz)."""

    if len(signal_data) < 30:
        return signal_data

    nyquist = fs / 2
    b, a    = sp_signal.butter(3, [low / nyquist, high / nyquist], btype='band')

    try:
        filtered = sp_signal.filtfilt(b, a, signal_data)
    except Exception:
        filtered = signal_data

    return filtered

In [7]:
# ================================
# FIXED BPM COMPUTATION
# (corrects FFT frequency indexing)
# ================================

def compute_bpm(signal_data, fs=30):
    """Estimate BPM from rPPG signal via FFT peak in the heart-rate band."""

    signal_data = np.asarray(signal_data)

    if len(signal_data) < 30:
        return 0.0

    yf = np.abs(np.fft.rfft(signal_data))
    xf = np.fft.rfftfreq(len(signal_data), d=1.0 / fs)

    mask = (xf >= 0.7) & (xf <= 4.0)
    xf, yf = xf[mask], yf[mask]

    if len(yf) == 0:
        return 0.0

    peak_freq = xf[np.argmax(yf)]
    bpm       = peak_freq * 60.0

    return bpm

## 5. Dataset Preprocessing

In [ ]:
def preprocess_dataset():
    """Extract face ROIs and ground-truth signals; save as .npz per subject."""

    subjects = sorted(os.listdir(cfg.DATASET_PATH))

    for subject in tqdm(subjects):

        subject_path = os.path.join(cfg.DATASET_PATH, subject)
        video_path   = os.path.join(subject_path, "vid.avi")
        gt_path      = os.path.join(subject_path, "ground_truth.txt")

        if not os.path.exists(video_path) or not os.path.exists(gt_path):
            continue

        save_path = os.path.join(cfg.PREPROCESS_DIR, f"{subject}.npz")

        if os.path.exists(save_path):
            continue

        gt_signal = np.loadtxt(gt_path)
        cap       = cv2.VideoCapture(video_path)
        frames    = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            roi = extract_face_roi(frame)
            if roi is None:
                continue
            frames.append(roi.astype(np.float32) / 255.0)

        cap.release()

        frames    = np.array(frames)
        min_len   = min(len(frames), len(gt_signal))
        frames    = frames[:min_len]
        gt_signal = gt_signal[:min_len]

        np.savez(save_path, frames=frames, signal=gt_signal)


preprocess_dataset()

  0%|                                                    | 0/31 [00:00<?, ?it/s]

## 6. PyTorch Dataset & DataLoader

In [ ]:
class RPPGDataset(Dataset):

    def __init__(self, files):
        self.files = files

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        data   = np.load(self.files[idx])
        frames = data['frames']
        signal = data['signal']

        # Pad if shorter than SEQ_LEN, else random crop
        if len(frames) <= cfg.SEQ_LEN:
            pad    = cfg.SEQ_LEN - len(frames)
            frames = np.pad(frames, ((0, pad), (0, 0), (0, 0), (0, 0)))
            signal = np.pad(signal, (0, pad))
            start  = 0
        else:
            start  = random.randint(0, len(frames) - cfg.SEQ_LEN)

        frames = frames[start:start + cfg.SEQ_LEN]
        signal = signal[start:start + cfg.SEQ_LEN]

        # Bandpass + normalise ground-truth signal
        signal = bandpass_filter(signal)
        signal = (signal - signal.mean()) / max(signal.std(), 1e-6)

        bpm = compute_bpm(signal)

        frames = torch.tensor(frames).permute(0, 3, 1, 2).float()
        signal = torch.tensor(signal).float()
        bpm    = torch.tensor(bpm).float()

        return frames, signal, bpm

In [ ]:
# ================================
# IMPROVED DATALOADER SETTINGS
# (num_workers=0 avoids multiprocessing issues on Windows/macOS)
# ================================

all_files = [
    os.path.join(cfg.PREPROCESS_DIR, x)
    for x in os.listdir(cfg.PREPROCESS_DIR)
    if x.endswith(".npz")
]

train_files, val_files = train_test_split(
    all_files,
    test_size=0.2,
    random_state=42
)

train_dataset = RPPGDataset(train_files)
val_dataset   = RPPGDataset(val_files)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Train batches     :", len(train_loader))
print("Validation batches:", len(val_loader))

## 7. Model Architecture

In [ ]:
# ================================
# POSITIONAL ENCODING
# ================================

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):

        super().__init__()

        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
# ================================
# IMPROVED SPATIAL ENCODER
# (BatchNorm + Dropout added for stability)
# ================================

class ImprovedSpatialEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Dropout(0.2),

            nn.AdaptiveAvgPool2d(1)
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        x = self.encoder(x)
        x = x.squeeze(-1).squeeze(-1)   # (B*T, 64)
        x = x.view(B, T, cfg.D_MODEL)   # (B, T, D_MODEL)
        return x

In [ ]:
# ================================
# IMPROVED TRANSFORMER MODEL
# (temporal conv + positional encoding + BPM head restored)
# ================================

class ImprovedTransformerRPPG(nn.Module):

    def __init__(self):

        super().__init__()

        self.spatial = ImprovedSpatialEncoder()

        # Temporal 1-D conv before the Transformer
        self.temporal_conv = nn.Conv1d(
            cfg.D_MODEL, cfg.D_MODEL,
            kernel_size=3, padding=1
        )

        self.pos_encoder = PositionalEncoding(cfg.D_MODEL)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.D_MODEL,
            nhead=cfg.NHEAD,
            batch_first=True,
            dropout=0.1
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=cfg.NUM_LAYERS
        )

        # Waveform head: per-timestep prediction
        self.signal_head = nn.Linear(cfg.D_MODEL, 1)

        # BPM head: scalar prediction from pooled features
        self.bpm_head = nn.Sequential(
            nn.Linear(cfg.D_MODEL, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):

        x = self.spatial(x)                # (B, T, D_MODEL)

        x = x.permute(0, 2, 1)            # (B, D_MODEL, T)
        x = self.temporal_conv(x)         # temporal mixing
        x = x.permute(0, 2, 1)            # (B, T, D_MODEL)

        x = self.pos_encoder(x)
        x = self.transformer(x)           # (B, T, D_MODEL)

        signal  = self.signal_head(x).squeeze(-1)          # (B, T)
        pooled  = x.mean(dim=1)                            # (B, D_MODEL)
        bpm_out = self.bpm_head(pooled).squeeze(-1)        # (B,)

        return signal, bpm_out

## 8. Loss Functions & Metrics

In [ ]:
def negative_pearson(pred, target):
    """Negative Pearson correlation loss for rPPG waveform alignment."""

    pred   = pred   - pred.mean(dim=1, keepdim=True)
    target = target - target.mean(dim=1, keepdim=True)

    numerator   = torch.sum(pred * target, dim=1)
    denominator = torch.sqrt(
        torch.sum(pred ** 2, dim=1) * torch.sum(target ** 2, dim=1)
    )

    return (1 - numerator / (denominator + 1e-8)).mean()

In [ ]:
# ================================
# VALIDATION METRICS
# ================================

def compute_metrics(pred_signal, gt_signal):
    """Compute MAE, RMSE, and Pearson correlation between predicted and GT signals."""

    pred_signal = pred_signal.flatten()
    gt_signal   = gt_signal.flatten()

    mae     = np.mean(np.abs(pred_signal - gt_signal))
    rmse    = np.sqrt(np.mean((pred_signal - gt_signal) ** 2))
    pearson = np.corrcoef(pred_signal, gt_signal)[0, 1]

    return {"MAE": mae, "RMSE": rmse, "Pearson": pearson}

## 9. Model Initialisation

In [ ]:
model = ImprovedTransformerRPPG().to(cfg.DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=cfg.LR)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    patience=3,
    verbose=True
)

# ================================
# SAFE GRADSCALER
# (device-aware constructor for PyTorch >= 2.3)
# ================================
scaler = GradScaler(
    device=cfg.DEVICE,
    enabled=torch.cuda.is_available()
)

print("CUDA Available:", torch.cuda.is_available())
print("Model parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 10. Training Loop

In [ ]:
# ================================
# TRAINING + VALIDATION LOOP
# (fixed model output unpacking, added validation, scheduler step)
# ================================

best_val_loss = float('inf')
history       = {"train_loss": [], "val_loss": []}

for epoch in range(cfg.EPOCHS):

    # ---- Training ----
    model.train()
    train_loss = 0.0

    for frames, signal, bpm in tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.EPOCHS} [train]"):

        frames = frames.to(cfg.DEVICE)
        signal = signal.to(cfg.DEVICE)
        bpm    = bpm.to(cfg.DEVICE)

        optimizer.zero_grad()

        with autocast(device_type=cfg.DEVICE, enabled=torch.cuda.is_available()):

            # FIX: model now correctly returns (signal, bpm) — both heads used
            pred_signal, pred_bpm = model(frames)

            waveform_loss = negative_pearson(pred_signal, signal)
            bpm_loss      = nn.functional.l1_loss(pred_bpm, bpm)

            loss = waveform_loss + 0.1 * bpm_loss

        scaler.scale(loss).backward()

        # Gradient clipping prevents exploding gradients
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ---- Validation ----
    model.eval()
    val_loss       = 0.0
    all_pred_np    = []
    all_gt_np      = []

    with torch.no_grad():
        for frames, signal, bpm in tqdm(val_loader, desc=f"Epoch {epoch+1}/{cfg.EPOCHS} [val]  "):

            frames = frames.to(cfg.DEVICE)
            signal = signal.to(cfg.DEVICE)
            bpm    = bpm.to(cfg.DEVICE)

            with autocast(device_type=cfg.DEVICE, enabled=torch.cuda.is_available()):
                pred_signal, pred_bpm = model(frames)
                waveform_loss = negative_pearson(pred_signal, signal)
                bpm_loss      = nn.functional.l1_loss(pred_bpm, bpm)
                loss          = waveform_loss + 0.1 * bpm_loss

            val_loss += loss.item()

            all_pred_np.append(pred_signal.cpu().numpy())
            all_gt_np.append(signal.cpu().numpy())

    val_loss /= len(val_loader)

    # Compute metrics on validation set
    metrics = compute_metrics(
        np.concatenate(all_pred_np),
        np.concatenate(all_gt_np)
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    # Step the LR scheduler on validation loss
    scheduler.step(val_loss)

    print(
        f"Epoch [{epoch+1:02d}/{cfg.EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"MAE: {metrics['MAE']:.4f} | "
        f"RMSE: {metrics['RMSE']:.4f} | "
        f"Pearson: {metrics['Pearson']:.4f}"
    )

    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_rppg_model.pth")
        print("  → Best model saved.")

print("\nTraining complete. Best val loss:", round(best_val_loss, 4))

## 11. Loss Curve Visualisation

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"],   label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("rPPG Training History")
plt.legend()
plt.tight_layout()
plt.savefig("training_curve.png", dpi=150)
plt.show()